# TradeFlow AI — nb0_real_doc_augmentation

**Tujuan**: 8 PDF carrier asli → 400+ gambar augmented sebagai domain adaptation signal  
**Prinsip anti-leakage**: Ground-truth label TIDAK digunakan — ini adalah augmentasi visual murni  
**Split**: 5 docs → train (255 gambar) | 3 docs → val (63 gambar) | 8 original → eval (tidak pernah disentuh)

---
### Mengapa Notebook Ini Ada?
Kita hanya punya 8 dokumen asli. Jika dipakai untuk training langsung:
1. Jumlahnya terlalu sedikit (model butuh ribuan sampel)
2. Model akan hafal 8 dokumen itu (overfitting)
3. nb4_eval jadi tidak bermakna (soal ujian = soal latihan)

Solusi: Kita augmentasi ke 400+ gambar, pakai sebagai **signal early-stopping** saja —  
bukan sebagai training data utama. Ground truth JSON tidak pernah disentuh di sini.

In [ ]:
# Install di Kaggle T4 — jalankan sekali
!pip install -q pdf2image pillow albumentations==1.3.1 opencv-python-headless
!apt-get install -qq poppler-utils  # diperlukan pdf2image

In [ ]:
import os, json, random
import numpy as np
from pathlib import Path
from pdf2image import convert_from_path
from PIL import Image, ImageDraw, ImageFont
import cv2
import albumentations as A
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)
print('Imports OK')

## 1. Konfigurasi Path & Split

In [ ]:
# ── Path config untuk Kaggle ──────────────────────────────────────────────
# Pastikan Anda sudah membuat dataset Kaggle bernama 'tradeflow-real-docs'
# dan mengupload ke-8 file PDF B/L asli ke dalamnya.
FIXTURES_DIR      = Path('/kaggle/input/tradeflow-real-docs')
OUTPUT_DIR        = Path('./dataset/augmented')
MANIFEST_PATH     = Path('./dataset/augmented_manifest.json')
AUG_PER_TRAIN_DOC = 50   # 5 docs × 50 = 250 augmented images
AUG_PER_VAL_DOC   = 20   # 3 docs × 20 =  60 augmented images
DPI               = 200  # DPI rasterisasi

# 5/3 train-val split — TIDAK menggunakan GT labels di sini!
TRAIN_DOCS = [
    'Hapag Filled 1.pdf',
    'Maersk Filled 1.pdf',
    'Evergreen Filled 1.pdf',
    'Evergreen Filled 2.pdf',
    'Cordelia Filled 1.pdf',
]
VAL_DOCS = [
    'Hapag Filled 2.pdf',
    'Evergreen Filled 3.pdf',
    'MSC Filled 1.pdf',
]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Train docs  : {len(TRAIN_DOCS)}')
print(f'Val docs    : {len(VAL_DOCS)}')
print(f'Expected images: {len(TRAIN_DOCS)*(AUG_PER_TRAIN_DOC+1) + len(VAL_DOCS)*(AUG_PER_VAL_DOC+1)}')

## 2. Fungsi Augmentasi

10 transformasi mensimulasikan degradasi scan dunia nyata yang kita lihat di dokumen asli.

In [ ]:
WATERMARKS = ['DRAFT', 'ORIGINAL', 'PROOFREAD', 'READ']

def add_watermark(img_np: np.ndarray, text: str) -> np.ndarray:
    """Watermark diagonal semi-transparan (sesuai pola carrier di CLAUDE.md)."""
    img = Image.fromarray(img_np).convert('RGBA')
    overlay = Image.new('RGBA', img.size, (255, 255, 255, 0))
    draw = ImageDraw.Draw(overlay)
    font_size = max(40, img.width // 8)
    try:
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', font_size)
    except Exception:
        font = ImageFont.load_default()
    alpha = random.randint(25, 65)  # sangat transparan
    x = random.randint(img.width // 6, img.width // 3)
    y = random.randint(img.height // 4, img.height // 2)
    draw.text((x, y), text, font=font, fill=(110, 110, 110, alpha))
    img = Image.alpha_composite(img, overlay)
    return np.array(img.convert('RGB'))


def add_shadow(img: np.ndarray) -> np.ndarray:
    """Simulasi bayangan scan fisik di salah satu tepi."""
    out = img.astype(np.float32)
    h, w = out.shape[:2]
    side = random.choice(['L', 'R', 'T', 'B'])
    width = random.randint(w // 12, w // 5)
    ramp = np.linspace(0.50, 1.0, width).astype(np.float32)
    if   side == 'L': out[:, :width]  *= ramp[None, :, None]
    elif side == 'R': out[:, -width:] *= ramp[::-1][None, :, None]
    elif side == 'T': out[:width, :]  *= ramp[:, None, None]
    else:             out[-width:, :] *= ramp[::-1][:, None, None]
    return out.clip(0, 255).astype(np.uint8)


def add_fold_line(img: np.ndarray) -> np.ndarray:
    """Lipatan halus horizontal atau vertikal."""
    out = img.copy()
    h, w = out.shape[:2]
    if random.random() < 0.5:
        y = random.randint(h // 5, 4 * h // 5)
        out[y:y+2] = np.clip(out[y:y+2].astype(int) - 40, 0, 255)
    else:
        x = random.randint(w // 5, 4 * w // 5)
        out[:, x:x+2] = np.clip(out[:, x:x+2].astype(int) - 40, 0, 255)
    return out


# Albumentations pipeline dasar
_aug_pipeline = A.Compose([
    A.Rotate(limit=3, border_mode=cv2.BORDER_CONSTANT, value=255, p=0.8),
    A.GaussianBlur(blur_limit=(3, 7), sigma_limit=(0.1, 1.5), p=0.5),
    A.ImageCompression(quality_lower=62, quality_upper=95, p=0.65),
    A.GaussNoise(var_limit=(10.0, 180.0), mean=0, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.6),
    A.Perspective(scale=(0.005, 0.025), p=0.3),
    A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.2),
    A.Sharpen(alpha=(0.05, 0.2), lightness=(0.9, 1.1), p=0.15),
])


def augment_image(img_np: np.ndarray) -> np.ndarray:
    """Terapkan stack augmentasi random ke satu gambar B/L."""
    img = _aug_pipeline(image=img_np)['image']
    if random.random() < 0.35:
        img = add_watermark(img, random.choice(WATERMARKS))
    if random.random() < 0.30:
        img = add_shadow(img)
    if random.random() < 0.20:
        img = add_fold_line(img)
    return img

print('Fungsi augmentasi siap')

## 3. Proses Setiap Dokumen Asli

In [ ]:
manifest = {'train': [], 'val': []}

# Halaman yang di-SKIP per carrier (CLAUDE.md §10: T&C + demurrage pages)
SKIP_PAGES = {
    'Cordelia Filled 1': [2],  # Klausul 20-pasal T&C di halaman 2
    'Hapag Filled 2':    [3],  # Klausul demurrage di halaman 3
}


def process_doc(pdf_name: str, split: str, n_aug: int) -> int:
    """Rasterisasi halaman 1 PDF, augmentasi n_aug kali, simpan, return jumlah gambar."""
    pdf_path = FIXTURES_DIR / pdf_name
    if not pdf_path.exists():
        print(f'  [SKIP — tidak ditemukan] {pdf_path}')
        return 0

    doc_id  = Path(pdf_name).stem
    out_dir = OUTPUT_DIR / doc_id
    out_dir.mkdir(parents=True, exist_ok=True)

    # Selalu rasterisasi halaman 1 saja (halaman form carrier, bukan T&C)
    pages = convert_from_path(str(pdf_path), dpi=DPI, first_page=1, last_page=1)
    page_np = np.array(pages[0].convert('RGB'))

    # Simpan original (tidak di-augment)
    orig_path = out_dir / 'orig.jpg'
    Image.fromarray(page_np).save(str(orig_path), quality=95)
    manifest[split].append({'path': str(orig_path), 'doc_id': doc_id, 'aug_n': 0, 'split': split})

    # Generate varian augmented
    for n in range(1, n_aug + 1):
        aug = augment_image(page_np)
        aug_path = out_dir / f'aug_{n:03d}.jpg'
        Image.fromarray(aug).save(str(aug_path), quality=90)
        manifest[split].append({'path': str(aug_path), 'doc_id': doc_id, 'aug_n': n, 'split': split})

    total = 1 + n_aug
    print(f'  [{split:5}] {doc_id:35} → {total:3d} gambar')
    return total


print('Memproses TRAIN docs...')
for doc in TRAIN_DOCS:
    process_doc(doc, 'train', AUG_PER_TRAIN_DOC)

print('\nMemproses VAL docs...')
for doc in VAL_DOCS:
    process_doc(doc, 'val', AUG_PER_VAL_DOC)

MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))

print(f'\n✓ Train images : {len(manifest["train"])}')
print(f'✓ Val images   : {len(manifest["val"])}')
print(f'✓ Manifest     : {MANIFEST_PATH}')

## 4. Verifikasi Visual

In [ ]:
# Tampilkan 4 crop augmented dari dokumen training pertama
first_doc_id  = Path(TRAIN_DOCS[0]).stem
first_doc_dir = OUTPUT_DIR / first_doc_id
sample_paths  = sorted(first_doc_dir.glob('aug_*.jpg'))[:4]

if sample_paths:
    fig, axes = plt.subplots(1, 4, figsize=(22, 6))
    for ax, p in zip(axes, sample_paths):
        img = Image.open(p)
        w, h = img.size
        ax.imshow(img.crop((0, 0, w, h // 3)))  # sepertiga atas (area header)
        ax.set_title(f'{p.parent.name}\n{p.name}', fontsize=7)
        ax.axis('off')
    plt.suptitle(f'Contoh augmentasi: {first_doc_id} (crop sepertiga atas)', fontsize=11)
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / 'augmentation_samples.png'), dpi=100, bbox_inches='tight')
    plt.show()
    print('Tersimpan: augmentation_samples.png')

## Ringkasan

| Split | Docs | Gambar | Digunakan di |
|---|---|---|---|
| **Train** | 5 | 255 | nb3 Phase 0 DAPT + Phase 2 mixing |
| **Val** | 3 | 63 | nb3 early-stopping ANLS signal |
| **Test** | 8 originals | 8 | nb4_eval.ipynb SAJA |

> ✅ Tidak ada ground-truth label yang dikonsumsi di sini — zero leakage risk.

**Lanjutkan ke nb1_synthetic_generator.ipynb →**